# Milestone 2 Lab, notebook 5 of 6: An agent that tests and fixes its own code

## Where we are

| Notebook | What you do |
|---|---|
| 1. Where we are, and the business's records | Recall the question and the answer key, and explore the records profit is calculated from |
| 2. One calculation, two engines | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| 3. A model drafts a transformation | Ask a model to write the calculation, and let the tests judge it |
| 4. Which model? Accuracy, cost, and speed | Give the same tasks to several models and compare them |
| **5. An agent that tests and fixes its own code** (you are here) | Give a model the tests as a tool, and compare it with a single draft |
| 6. The pipeline, and the same question again | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **about 20**. A full run of this lab costs well under a dollar.

## What this notebook is for

In notebook 4 each model had one shot. If its draft failed a test, that was that.

A person doesn't work that way. They run the tests, read what failed, fix the code, and run the tests again. In milestone 1 you learned that an **agent** is a model, with tools, in a loop. So here we give a model one tool, `run_tests`, and let it work the way a person does.

Two questions for this notebook:

1. Can a model **repair** a draft that fails the tests?
2. Can an inexpensive model with a test loop match an expensive model's single draft?

In [1]:
import os

import pandas as pd
from lab_helpers import (
    OPENROUTER_BASE_URL,
    broken_all_in_one_sql,
    cases_table,
    connect,
    repeat,
    save_result,
    show_code,
    show_doc,
)
from langchain_openai import ChatOpenAI

from retail_profit_stack.transform.drafting import (
    AGENT_INSTRUCTIONS,
    draft_and_score,
    run_draft,
    run_fix_agent,
)
from retail_profit_stack.transform.specs import TASKS

client, MODEL = connect(timeout=90, max_retries=1)

CHEAP_MODEL = "google/gemma-4-31b-it"
STRONG_MODEL = "anthropic/claude-sonnet-5.5"
TASK = TASKS["profit_all_in_one"]
LANGUAGE = "sql"

Connected to OpenRouter. Default model: anthropic/claude-haiku-5.5


`CHEAP_MODEL` is a small, inexpensive model. `STRONG_MODEL` is a more capable one that costs many times more. The task is the hard one: the whole profit calculation in a single query.

If you ran notebook 4, look at its summary and pick your own two models.

## Step 1: The design

In [2]:
show_doc("LLD.md", "11. The agent that fixes its own draft")

> **From `docs/LLD.md`: 11. The agent that fixes its own draft**

An agent built with LangChain's `create_agent`, as in milestone 1, with one tool:

```python
def run_tests(code: str) -> str
    # Runs the draft against the hand-worked cases with the limits in section 9.
    # Returns how many passed and, for each failure, the expected and actual rows.
```

The agent is given the same task prompt and told to call `run_tests` until every case passes, up to 5 attempts. It can also be handed an existing draft to repair (`starting_code`): that draft is tested first and recorded as attempt 1. `AgentDraftRun` records the final code, each attempt, whether every case passed, and the total tokens, cost, and time.

The lab uses it twice. First it hands the agent a draft with a known mistake (order lines joined straight to refunds) and watches the repair. Then it compares three ways of writing the all-in-one task from scratch: an inexpensive model's single draft, the same model with the tests as a tool, and a stronger model's single draft. Because models do not give the same answer twice, each is run three times.

A loop helps only a model that can act on a failure message. In trials when the lab was built, one small model went from 5 correct runs in 6 to 6 in 6 with the tests, while another repeated the same error on all five attempts and cost six times as much as a single draft. The attempt limit exists for that case.

Code from the agent is treated like any other draft. Passing the tests makes it a candidate. A person reviews it before it replaces a production implementation.

## Step 2: A draft with a mistake in it

Start with a draft we know is wrong. This is the all-in-one query with the mistake from notebook 2 put into it: it joins order lines straight to refunds.

Read it. Can you find the problem? It is one line, and the query runs without any error.

In [3]:
broken = broken_all_in_one_sql()

show_code(broken, "sql")

```sql
WITH sales AS (
    SELECT strftime(o.order_date, '%Y-%m') AS month, i.product_id,
           SUM(i.quantity) AS units_sold,
           SUM(i.quantity * i.unit_list_price) AS gross_sales,
           SUM(i.discount_amount) AS discounts
    FROM order_items AS i JOIN orders AS o ON o.order_id = i.order_id
    LEFT JOIN refunds AS x ON x.order_item_id = i.order_item_id
    WHERE o.status <> 'cancelled'
    GROUP BY 1, 2
), refund_totals AS (
    SELECT strftime(refund_date, '%Y-%m') AS month, product_id,
           SUM(refund_amount) AS refunds, SUM(return_handling_cost) AS return_cost
    FROM refunds GROUP BY 1, 2
), delivery_costs AS (
    SELECT strftime(ship_date, '%Y-%m') AS month, product_id,
           SUM(pick_pack_cost) AS fulfillment_cost, SUM(carrier_cost) AS shipping_cost
    FROM shipments GROUP BY 1, 2
), fee_totals AS (
    SELECT strftime(capture_date, '%Y-%m') AS month, product_id, SUM(fee_amount) AS payment_fees
    FROM payment_fees GROUP BY 1, 2
), joined AS (
    SELECT s.month, s.product_id, s.units_sold, s.gross_sales, s.discounts,
           COALESCE(r.refunds, 0) AS refunds,
           s.units_sold * c.unit_cost AS product_cost,
           COALESCE(d.fulfillment_cost, 0) AS fulfillment_cost,
           COALESCE(d.shipping_cost, 0) AS shipping_cost,
           COALESCE(f.payment_fees, 0) AS payment_fees,
           COALESCE(r.return_cost, 0) AS return_cost
    FROM sales AS s
    LEFT JOIN refund_totals AS r ON r.month = s.month AND r.product_id = s.product_id
    LEFT JOIN delivery_costs AS d ON d.month = s.month AND d.product_id = s.product_id
    LEFT JOIN fee_totals AS f ON f.month = s.month AND f.product_id = s.product_id
    LEFT JOIN product_costs AS c ON c.month = s.month AND c.product_id = s.product_id
)
SELECT month, product_id, units_sold, gross_sales, discounts, refunds,
       gross_sales - discounts - refunds AS net_sales,
       product_cost,
       gross_sales - discounts - refunds - product_cost AS gross_profit,
       fulfillment_cost, shipping_cost, payment_fees, return_cost,
       gross_sales - discounts - refunds - product_cost
           - fulfillment_cost - shipping_cost - payment_fees - return_cost AS contribution_profit,
       '1.0.0' AS metric_version
FROM joined
```

The tests find it at once.

In [14]:
cases_table(run_draft(broken, LANGUAGE, TASK))

,case,passed,what went wrong
0,one plain order,yes,
1,everything at once,NO,"row 3, units_sold: expected 2, got 4 (row: {'m..."
2,a line loaded twice,yes,


The case **"everything at once"** fails: it expected 2 units and got 4. That case has a line with two refunds, so the join counted the line twice. Notice that the other two cases pass. A query can be wrong and still look right on simple data.

## Step 3: Hand it to an agent

Here are the agent's instructions. They describe how a careful engineer works.

In [5]:
print(AGENT_INSTRUCTIONS)

You are a data engineer. You will be asked to write one transformation.

You have a tool, run_tests. Give it your code and it runs the hand-worked test cases and tells
you which failed, with the expected and the actual values.

Work like this:
1. Write the code and call run_tests with it.
2. If any case failed, read the failure, fix the code, and call run_tests again.
3. Stop as soon as every case passes, or when the tool says you have no attempts left.

You have 5 attempts. Always pass the complete code to run_tests, never a fragment.
When you finish, reply with one short sentence saying whether the tests pass.



The agent has one tool, `run_tests`. It takes code, runs it against the hand-worked cases under the same limits as before, and replies with what failed: the case name, and the expected and actual values.

That reply is the important part. "Expected 2, got 4 for units_sold" is something a model can act on.

`run_fix_agent` builds the agent with LangChain's `create_agent`, as you did in milestone 1. We give it the broken draft and the test report, and let it work. Every attempt is recorded. Attempt 1 is the broken draft itself.

In [6]:
cheap_llm = ChatOpenAI(model=CHEAP_MODEL, base_url=OPENROUTER_BASE_URL, api_key=os.getenv("OPENROUTER_API_KEY"), temperature=0, timeout=120, max_retries=2)

repair = run_fix_agent(cheap_llm, TASK, LANGUAGE, CHEAP_MODEL, starting_code=broken)

print(f"Every case passed: {repair.all_passed}.  Time: {repair.seconds} seconds")
pd.DataFrame([{"attempt": attempt.number, "cases passed": f"{attempt.cases_passed} of {attempt.cases_total}"} for attempt in repair.attempts])

Every case passed: True.  Time: 20.2 seconds


,attempt,cases passed
0,1,2 of 3
1,2,3 of 3


Read down the attempts. The first row is the broken draft. Each row after it is the model's own fix, tested. You are watching a model read a failure and correct the code, with no person involved.

> **If there is only one attempt**, the model did not call the tool. Run the cell again, or choose a different `CHEAP_MODEL`: not every small model is good at using tools.

Here is what it ended with. Find the change.

In [7]:
show_code(repair.final_code, LANGUAGE)

```sql
WITH sales AS (
    SELECT strftime(o.order_date, '%Y-%m') AS month, i.product_id,
           SUM(i.quantity) AS units_sold,
           SUM(i.quantity * i.unit_list_price) AS gross_sales,
           SUM(i.discount_amount) AS discounts
    FROM order_items AS i 
    JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY 1, 2
), refund_totals AS (
    SELECT strftime(refund_date, '%Y-%m') AS month, product_id,
           SUM(refund_amount) AS refunds, SUM(return_handling_cost) AS return_cost
    FROM refunds GROUP BY 1, 2
), delivery_costs AS (
    SELECT strftime(ship_date, '%Y-%m') AS month, product_id,
           SUM(pick_pack_cost) AS fulfillment_cost, SUM(carrier_cost) AS shipping_cost
    FROM shipments GROUP BY 1, 2
), fee_totals AS (
    SELECT strftime(capture_date, '%Y-%m') AS month, product_id, SUM(fee_amount) AS payment_fees
    FROM payment_fees GROUP BY 1, 2
), all_keys AS (
    SELECT month, product_id FROM sales
    UNION
    SELECT month, product_id FROM refund_totals
    UNION
    SELECT month, product_id FROM delivery_costs
    UNION
    SELECT month, product_id FROM fee_totals
), joined AS (
    SELECT 
        k.month, 
        k.product_id, 
        COALESCE(s.units_sold, 0) AS units_sold, 
        COALESCE(s.gross_sales, 0.00) AS gross_sales, 
        COALESCE(s.discounts, 0.00) AS discounts,
        COALESCE(r.refunds, 0.00) AS refunds,
        COALESCE(d.fulfillment_cost, 0.00) AS fulfillment_cost,
        COALESCE(d.shipping_cost, 0.00) AS shipping_cost,
        COALESCE(f.payment_fees, 0.00) AS payment_fees,
        COALESCE(r.return_cost, 0.00) AS return_cost,
        c.unit_cost
    FROM all_keys k
    LEFT JOIN sales s ON s.month = k.month AND s.product_id = k.product_id
    LEFT JOIN refund_totals r ON r.month = k.month AND r.product_id = k.product_id
    LEFT JOIN delivery_costs d ON d.month = k.month AND d.product_id = k.product_id
    LEFT JOIN fee_totals f ON f.month = k.month AND f.product_id = k.product_id
    LEFT JOIN product_costs c ON c.month = k.month AND c.product_id = k.product_id
)
SELECT 
    month, 
    product_id, 
    units_sold, 
    gross_sales, 
    discounts, 
    refunds,
    (gross_sales - discounts - refunds) AS net_sales,
    (units_sold * unit_cost) AS product_cost,
    (gross_sales - discounts - refunds) - (units_sold * unit_cost) AS gross_profit,
    fulfillment_cost, 
    shipping_cost, 
    payment_fees, 
    return_cost,
    (gross_sales - discounts - refunds) - (units_sold * unit_cost) - fulfillment_cost - shipping_cost - payment_fees - return_cost AS contribution_profit,
    '1.0.0' AS metric_version
FROM joined
```

We don't take the agent's word for it. Run its final code through the tests ourselves.

In [8]:
cases_table(run_draft(repair.final_code, LANGUAGE, TASK))

,case,passed,what went wrong
0,one plain order,yes,
1,everything at once,yes,
2,a line loaded twice,yes,


## Step 4: One shot, or a loop?

Now the second question. Three ways to get this transformation written from scratch:

1. the cheap model, one draft
2. the cheap model as an agent, with the tests as a tool
3. the strong model, one draft

Models don't give the same answer twice, so one run of each would tell us very little. A single result is an anecdote. We run each approach three times and count how often it gets every case right.

In [9]:
REPEATS = 3

one_shots = repeat(lambda: draft_and_score(client, CHEAP_MODEL, TASK, LANGUAGE), REPEATS)
agent_runs = repeat(lambda: run_fix_agent(cheap_llm, TASK, LANGUAGE, CHEAP_MODEL), REPEATS)
strong_shots = repeat(lambda: draft_and_score(client, STRONG_MODEL, TASK, LANGUAGE), REPEATS)

print("Cheap, one draft:   cases passed in each run:", [result.cases_passed for result in one_shots])
print("Cheap, with tests:  attempts in each run:    ", [[attempt.cases_passed for attempt in run.attempts] for run in agent_runs])
print("Strong, one draft:  cases passed in each run:", [result.cases_passed for result in strong_shots])

Cheap, one draft:   cases passed in each run: [3, 3, 3]
Cheap, with tests:  attempts in each run:     [[3], [3], [3]]
Strong, one draft:  cases passed in each run: [3, 3, 3]


In the middle line, each inner list is one run: the cases passed on each attempt. `[3]` means it passed first time. `[1, 3]` means it failed, read the failure, and fixed it.

Side by side, with what each cost.

In [10]:
def row(approach, runs, attempts):
    return {
        "approach": approach,
        "fully correct": f"{sum(run.all_passed for run in runs)} of {len(runs)} runs",
        "attempts per run": round(sum(attempts) / len(runs), 1),
        "cost per run ($)": round(sum(run.cost for run in runs) / len(runs), 5),
        "seconds per run": round(sum(run.seconds for run in runs) / len(runs), 1),
    }


pd.DataFrame([
    row(f"Cheap model, one draft ({CHEAP_MODEL.split('/')[-1]})", one_shots, [1] * REPEATS),
    row("Cheap model, with the tests", agent_runs, [len(run.attempts) for run in agent_runs]),
    row(f"Strong model, one draft ({STRONG_MODEL.split('/')[-1]})", strong_shots, [1] * REPEATS),
])

,approach,fully correct,attempts per run,cost per run ($),seconds per run
0,"Cheap model, one draft (gemma-4-31b-it)",3 of 3 runs,1.0,0.00064,15.7
1,"Cheap model, with the tests",3 of 3 runs,1.0,0.00069,4.6
2,"Strong model, one draft (claude-sonnet-5.5)",3 of 3 runs,1.0,0.02286,14.1


What to look for:

- **Fully correct.** A single draft from the cheap model is usually right, but not always. With the tests as a tool, its code has been run and checked before it is handed over, and a failure gets another attempt.
- **Cost per run.** Compare the cheap model with the tests against the strong model's single draft. The difference is typically more than tenfold.
- **Seconds per run.**

Three runs is still a small sample, so your numbers will move from run to run. Raise `REPEATS` if you want firmer figures.

### A loop is not magic

When this lab was built, we tried the same experiment with several small models, six runs each. One was fully correct 5 times out of 6 in a single draft, and 6 out of 6 with the tests. Another was *no better* with the tests: when its first attempt failed, it made the same mistake five times running, and cost six times as much as a single draft.

A test loop only helps if the model can understand a failure message and change its code in response. Some can't. That is why the agent has a limit of five attempts, and why you measure before you rely on it. Change `CHEAP_MODEL` to `"mistralai/mistral-small-2603"` and run this notebook again to see a model that sometimes gets stuck.

## Step 5: Passing the tests is not the end

The agent's code passed every case. Does it go straight into production?

No. Passing the tests makes a draft a **candidate**. The tests check the cases someone thought of. A person still reads the code before it is trusted with the business's numbers, and compares it with what is already there. Here is the production implementation of the same task.

In [11]:
from retail_profit_stack.transform.engines.base import IMPLEMENTATIONS_DIR

show_code((IMPLEMENTATIONS_DIR / "sql" / "profit_all_in_one.sql").read_text(encoding="utf-8"), "sql")

```sql
-- profit_all_in_one: the whole calculation in one query (a lab task, not run by the pipeline).
-- The rule that makes it correct: sum each table on its own first, then join the sums.
-- Joining order lines straight to refunds or shipments would repeat rows and inflate totals.
WITH sales AS (
    SELECT strftime(o.order_date, '%Y-%m') AS month, i.product_id,
           SUM(i.quantity) AS units_sold,
           SUM(i.quantity * i.unit_list_price) AS gross_sales,
           SUM(i.discount_amount) AS discounts
    FROM order_items AS i JOIN orders AS o ON o.order_id = i.order_id
    WHERE o.status <> 'cancelled'
    GROUP BY 1, 2
), refund_totals AS (
    SELECT strftime(refund_date, '%Y-%m') AS month, product_id,
           SUM(refund_amount) AS refunds, SUM(return_handling_cost) AS return_cost
    FROM refunds GROUP BY 1, 2
), delivery_costs AS (
    SELECT strftime(ship_date, '%Y-%m') AS month, product_id,
           SUM(pick_pack_cost) AS fulfillment_cost, SUM(carrier_cost) AS shipping_cost
    FROM shipments GROUP BY 1, 2
), fee_totals AS (
    SELECT strftime(capture_date, '%Y-%m') AS month, product_id, SUM(fee_amount) AS payment_fees
    FROM payment_fees GROUP BY 1, 2
), joined AS (
    SELECT s.month, s.product_id, s.units_sold, s.gross_sales, s.discounts,
           COALESCE(r.refunds, 0) AS refunds,
           s.units_sold * c.unit_cost AS product_cost,
           COALESCE(d.fulfillment_cost, 0) AS fulfillment_cost,
           COALESCE(d.shipping_cost, 0) AS shipping_cost,
           COALESCE(f.payment_fees, 0) AS payment_fees,
           COALESCE(r.return_cost, 0) AS return_cost
    FROM sales AS s
    LEFT JOIN refund_totals AS r ON r.month = s.month AND r.product_id = s.product_id
    LEFT JOIN delivery_costs AS d ON d.month = s.month AND d.product_id = s.product_id
    LEFT JOIN fee_totals AS f ON f.month = s.month AND f.product_id = s.product_id
    LEFT JOIN product_costs AS c ON c.month = s.month AND c.product_id = s.product_id
)
SELECT month, product_id, units_sold, gross_sales, discounts, refunds,
       gross_sales - discounts - refunds AS net_sales,
       product_cost,
       gross_sales - discounts - refunds - product_cost AS gross_profit,
       fulfillment_cost, shipping_cost, payment_fees, return_cost,
       gross_sales - discounts - refunds - product_cost
           - fulfillment_cost - shipping_cost - payment_fees - return_cost AS contribution_profit,
       '1.0.0' AS metric_version
FROM joined
```

Compare it with the agent's version. Does the agent's sum each table before joining? Does it keep the product with no cost? Is it something you would be happy to maintain? Those are a reviewer's questions, and no test asks them.

## What we learned

- **Give an agent the tests and it can fix its own work.** The loop is the one you learned in milestone 1; only the tool is new.
- **A precise failure message is what makes the fix possible.** "Expected 2, got 4 for units_sold" is something a model can act on.
- **Reliability can be bought with a better model or with a loop.** Which is cheaper depends on the task and the model, and a loop does not rescue a model that cannot act on a failure. Run it several times and measure.
- **Tests gate, people review.** The model writes, the tests decide whether it is a candidate, and a person decides whether it ships.

**Next: notebook 6**, where the reviewed transformations run as a pipeline and the leader's question is asked again.

In [12]:
save_result("agent_run", {
    "cheap_model": CHEAP_MODEL, "strong_model": STRONG_MODEL,
    "repair_attempts": [attempt.cases_passed for attempt in repair.attempts], "repair_passed": repair.all_passed,
    "cheap_one_shot_passed": [result.all_passed for result in one_shots],
    "agent_passed": [run.all_passed for run in agent_runs],
    "strong_one_shot_passed": [result.all_passed for result in strong_shots],
})

Saved results/agent_run.json
